In [3]:
print("hello world")

hello world


In [10]:
import numpy as np, torch
a = np.array([[1, 2, 3], [4, 5, 6]])
t = torch.from_numpy(a)

In [12]:
print(t.shape)
print(t * 2)
print(t.float().mean())
print(t.numpy().shape)

torch.Size([2, 3])
tensor([[ 2,  4,  6],
        [ 8, 10, 12]])
tensor(3.5000)
(2, 3)


In [29]:
import torch, time 
def bench(device, n=1024, repeat=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    c = a @ b
    if device.type == "cuda": torch.cuda.synchronize()  
    t0 = time.perf_counter()
    for _ in range(repeat):
        c = a @ b
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeat * 1000

print(f"CPU : {bench(torch.device('cpu')):.2f} ms")
if torch.cuda.is_available():
    print(f"GPU : {bench(torch.device('cuda')):.2f} ms")

CPU : 3.79 ms
GPU : 6.45 ms


In [19]:
import torch, time

devs = [torch.device("cpu")]
if torch.cuda.is_available():
    devs.append(torch.device("cuda"))
for n in (256, 1024, 4096):
    print(n, " | ".join(f"{bench(d, n):7.2f} ms" for d in devs))
                             
                            

256    0.28 ms |    0.09 ms
1024    4.61 ms |    9.55 ms
4096  190.89 ms |   37.49 ms


In [23]:
import torch
y_pred = torch.tensor([2.5, 0.0, 2.0])
y_true = torch.tensor([3.0, -0.5, 2.0])
err = y_pred - y_true

print(err)
loss = (err ** 2).mean()

print(loss)

tensor([-0.5000,  0.5000,  0.0000])
tensor(0.1667)


In [37]:
import torch
x= torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
y.backward()
print(y.item(), x.grad)

w = torch.tensor([1.0, -1.0], requires_grad=True)
loss = (w * torch.tensor([2.0, 3.0])).sum()
loss.backward()
print(w.grad)
with torch.no_grad():
    w -= 0.1 * w.grad
print(w.detach())


15.0 tensor(8.)
tensor([2., 3.])
tensor([ 0.8000, -1.3000])


In [42]:
import torch
torch.manual_seed(0)
x = torch.linspace(0, 1, 50)
y = 2 * x + 1 + 0.05 * torch.randn(50)

w = torch.tensor(0., requires_grad=True)

b = torch.tensor(0., requires_grad=True)
for step in range(200):
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.5 * w.grad; b -= 0.5 * b.grad
        w.grad.zero_(); b.grad.zero_()
print(round(loss.item(), 4))
print(round(w.item(), 2), round(b.item(), 2))

0.0028
2.03 0.99


In [48]:
import torch
torch.manual_seed(0)
x = torch.linspace(0, 1, 50)
y = -3 * x + 0.5 + 0.05 * torch.randn(50)
w = torch.tensor(0., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
for step in range(300):
    loss = ((w * x + b -y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.5 * w.grad
        b -= 0.5 * b.grad
        w.grad.zero_()
        b.grad.zero_()
print(round(w.item(), 2))

-2.97
